# RQ2: Era-Moderated Regression on Real Apache JIRA Data

**Research Question 2:** Has the relationship between issue/process characteristics
(priority, number of comments) and defect resolution time changed between the
pre-AI-coding era (resolved before 2023-01-01) and the AI-assisted-coding era
(resolved 2023-01-01 onward)?

**Data:** 30,733 real, resolved Apache Software Foundation JIRA issues (Camel + Hadoop),
pulled via the public JIRA REST API and already cleaned in this repo's
`data/cleaned/qa_defect_dataset.csv` (era and resolution_time_days computed from real
timestamps).

**Note on scope:** `num_reassignments`, named as a predictor in the original synopsis,
was not captured by the JIRA extraction script and is not available here. This analysis
uses the two real predictors that ARE available: `num_comments` and `priority`.


In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import json

df = pd.read_csv("../data/cleaned/qa_defect_dataset.csv")
print(f"Loaded real Apache JIRA dataset: {len(df)} issues (CAMEL + HADOOP)")
print(df['project_name'].value_counts().to_dict())

Loaded real Apache JIRA dataset: 30733 issues (CAMEL + HADOOP)
{'CAMEL': 19927, 'HADOOP': 10806}


## Prepare the analysis sample

In [2]:
df = df.dropna(subset=["resolution_time_days", "num_comments", "priority", "era"])
df["era_binary"] = (df["era"] == "ai_era").astype(int)

print(f"Analysis sample after dropping missing: N = {len(df)}")
print(f"Pre-AI era: {(df['era_binary']==0).sum()}, AI era: {(df['era_binary']==1).sum()}")

Analysis sample after dropping missing: N = 30733
Pre-AI era: 25327, AI era: 5406


## RQ2: Moderated multiple regression

Full (moderated) model: `resolution_time_days ~ (num_comments + priority) * era`
Reduced model (no interaction): `resolution_time_days ~ num_comments + priority + era`

Comparing R² between these two models gives Cohen's f² for the era-interaction effect.

In [3]:
full_formula = "resolution_time_days ~ (num_comments + C(priority)) * era_binary"
full_model = smf.ols(full_formula, data=df).fit()
print(full_model.summary())

                             OLS Regression Results                             
Dep. Variable:     resolution_time_days   R-squared:                       0.030
Model:                              OLS   Adj. R-squared:                  0.030
Method:                   Least Squares   F-statistic:                     86.95
Date:                  Tue, 28 Jul 2026   Prob (F-statistic):          5.71e-195
Time:                          11:38:24   Log-Likelihood:            -2.1399e+05
No. Observations:                 30733   AIC:                         4.280e+05
Df Residuals:                     30721   BIC:                         4.281e+05
Df Model:                            11                                         
Covariance Type:              nonrobust                                         
                                         coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------

In [4]:
reduced_formula = "resolution_time_days ~ num_comments + C(priority) + era_binary"
reduced_model = smf.ols(reduced_formula, data=df).fit()

interaction_terms = [t for t in full_model.pvalues.index if ":era_binary" in t]
significant = [t for t in interaction_terms if full_model.pvalues[t] < 0.05]
print(f"Interaction terms tested: {interaction_terms}")
print(f"Significant Era interaction terms (p < .05): {significant}")

f2 = (full_model.rsquared - reduced_model.rsquared) / (1 - full_model.rsquared)
print(f"\nFull model R2 = {full_model.rsquared:.4f}, Reduced model R2 = {reduced_model.rsquared:.4f}")
print(f"Cohen's f2 (era-interaction effect size) = {f2:.4f}")

Interaction terms tested: ['C(priority)[T.Critical]:era_binary', 'C(priority)[T.Major]:era_binary', 'C(priority)[T.Minor]:era_binary', 'C(priority)[T.Trivial]:era_binary', 'num_comments:era_binary']
Significant Era interaction terms (p < .05): ['C(priority)[T.Critical]:era_binary', 'C(priority)[T.Major]:era_binary', 'C(priority)[T.Minor]:era_binary', 'C(priority)[T.Trivial]:era_binary']

Full model R2 = 0.0302, Reduced model R2 = 0.0291
Cohen's f2 (era-interaction effect size) = 0.0011


## Descriptive check: resolution time by era

In [5]:
med_by_era = df.groupby("era")["resolution_time_days"].agg(["median", "mean", "count"])
print(med_by_era)

        median       mean  count
era                             
ai_era     2.0  73.053644   5406
pre_ai     3.0  69.703360  25327


## Interpretation

All four priority × era interaction terms are statistically significant (p < .0001 for
Major/Minor/Trivial, p = .044 for Critical), meaning the priority-resolution time
relationship genuinely differs between eras in this real data. However, the practical
effect size (Cohen's f² ≈ 0.001) is negligible by Cohen's (1988) convention — a
statistically real but practically small effect, consistent with a very large N (30,733).

Descriptively, median resolution time is actually slightly *faster* in the AI-coding era
(2.0 days) than pre-AI (3.0 days), while mean resolution time is slightly higher in the
AI era — consistent with a heavy-tailed distribution rather than a uniform shift.

In [6]:
results = {
    "n_total": int(len(df)),
    "n_pre_ai": int((df['era_binary']==0).sum()),
    "n_ai_era": int((df['era_binary']==1).sum()),
    "full_r2": float(full_model.rsquared),
    "reduced_r2": float(reduced_model.rsquared),
    "f2": float(f2),
    "significant_interactions": significant,
}
with open("../data/cleaned/rq2_real_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("Saved rq2_real_results.json")
print(results)

Saved rq2_real_results.json
{'n_total': 30733, 'n_pre_ai': 25327, 'n_ai_era': 5406, 'full_r2': 0.030194067904357857, 'reduced_r2': 0.0291321904404106, 'f2': 0.0010949380992676132, 'significant_interactions': ['C(priority)[T.Critical]:era_binary', 'C(priority)[T.Major]:era_binary', 'C(priority)[T.Minor]:era_binary', 'C(priority)[T.Trivial]:era_binary']}
